# FairFace: huấn luyện các model trên Kaggle

Chạy lần lượt các cell. Notebook clone repository, tìm dataset FairFace có sẵn trong runtime (hoặc tải bằng Kaggle API), tạo train/validation/test split, rồi train các model đã chọn. Dataset cần có ảnh FairFace padding 0.25 và hai CSV train/validation (`train_labels.csv`/`val_labels.csv` hoặc `fairface_label_train.csv`/`fairface_label_val.csv`).

**Cấu hình quan trọng:** sửa `REPO_URL`, `DATA_DIR` (đường dẫn gốc dataset; để `None` để dò tự động) và `MODELS` ở cell cấu hình. Kaggle Input được mount read-only; output ghi vào thư mục working.


In [ ]:
# 1) Clone repository và cấu hình lần chạy
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/YOUR_USERNAME/DeepLearningV2.git"  # sửa URL repo
PROJECT_DIR = Path("/kaggle/working/DeepLearningV2")
DATA_DIR = None  # ví dụ: Path("/kaggle/input/fairface"); None = dò tự động
OUTPUT_DIR = Path("/kaggle/working/outputs")
MODELS = ["resnet50_base", "resnet50_head", "resnet50_se", "resnet50_cbam"]
EPOCHS = 25
SEED = 42

if not (PROJECT_DIR / "src" / "train.py").exists():
    if "YOUR_USERNAME" in REPO_URL:
        raise ValueError("Hãy sửa REPO_URL thành URL GitHub thật của repository.")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Project:", PROJECT_DIR)
print("Dataset override:", DATA_DIR)
print("Output:", OUTPUT_DIR)


## 2) Dataset

**Kaggle:** nhấn *Add Input* và chọn dataset FairFace (khuyến nghị để ảnh không phải tải lại mỗi phiên). Gán `DATA_DIR` thành thư mục chứa CSV và ảnh hoặc để `None` để dò dưới `/kaggle/input`.

Có thể tải dataset qua Kaggle API nếu runtime có credential `kaggle.json`/Kaggle Secrets: đặt `DOWNLOAD_KAGGLE_DATASET=True` và điền slug `owner/dataset`. Slug không được đoán sẵn vì các bản FairFace trên Kaggle khác nhau; cần chọn đúng bản padding 0.25. Trên Colab cần cấu hình Kaggle API token trước khi tải.

In [ ]:
# 2) Tùy chọn tải dataset bằng Kaggle API. Mặc định dùng dataset đã mount/đặt sẵn.
DOWNLOAD_KAGGLE_DATASET = False
KAGGLE_DATASET_SLUG = "owner/dataset-name"  # sửa nếu bật tải
DOWNLOAD_DIR = Path("/kaggle/working/data_downloads")

if DOWNLOAD_KAGGLE_DATASET:
    if KAGGLE_DATASET_SLUG == "owner/dataset-name":
        raise ValueError("Điền slug Kaggle dataset thật trước khi tải.")
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
    subprocess.run([sys.executable, "-m", "kaggle", "datasets", "download", "-d", KAGGLE_DATASET_SLUG,
                    "-p", str(DOWNLOAD_DIR), "--unzip"], check=True)
    DATA_DIR = DOWNLOAD_DIR

if DATA_DIR is None:
    DATA_DIR = Path("/kaggle/input")
else:
    DATA_DIR = Path(DATA_DIR)
if not DATA_DIR.exists():
    raise FileNotFoundError(f"DATA_DIR không tồn tại: {DATA_DIR}. Hãy đổi DATA_DIR hoặc thêm Kaggle Input.")
print("Dataset root:", DATA_DIR)


## 3) Cài thư viện và kiểm tra GPU

In [ ]:
# PyTorch/torchvision thường được cài sẵn trên Colab và Kaggle.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)
import torch, torchvision
print("PyTorch:", torch.__version__, "| torchvision:", torchvision.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))
else: print("Bật GPU trong Runtime/Settings trước khi train để tăng tốc.")


## 4) Kiểm tra dataset và tạo splits

In [ ]:
# Script kiểm tra CSV/đường dẫn ảnh rồi tạo train/validation từ train CSV;
# validation CSV gốc được giữ làm test. Splits lưu cạnh outputs.
subprocess.run([sys.executable, "scripts/prepare_splits.py", "--data_dir", str(DATA_DIR),
                "--out", str(OUTPUT_DIR.parent / "splits")], check=True)


## 5) Huấn luyện

Có thể chọn một model hoặc cả bốn. Lệnh `--resume` khôi phục checkpoint nếu phiên bị ngắt; để giảm thời gian thử nghiệm, giảm `EPOCHS`.

In [ ]:
for model_name in MODELS:
    cmd = [sys.executable, "-m", "src.train", "--model", model_name,
           "--config", "configs/default.yaml", "--seed", str(SEED),
           "--epochs", str(EPOCHS), "--resume", "--data_dir", str(DATA_DIR),
           "--output_dir", str(OUTPUT_DIR)]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)


## 6) Đánh giá và tải artifacts

Đánh giá best checkpoint trên test cho từng model. Trên Kaggle, artifacts nằm trong `/kaggle/working` và có thể tải từ Output sau khi chạy notebook.

In [ ]:
for model_name in MODELS:
    cmd = [sys.executable, "-m", "src.evaluate", "--model", model_name,
           "--seed", str(SEED), "--data_dir", str(DATA_DIR), "--output_dir", str(OUTPUT_DIR)]
    subprocess.run(cmd, check=True)

ARCHIVE = OUTPUT_DIR.parent / "fairface_outputs.zip"
shutil.make_archive(str(ARCHIVE.with_suffix("")), "zip", root_dir=OUTPUT_DIR)
print("Artifacts:", OUTPUT_DIR)
print("Archive:", ARCHIVE)
